<a href="https://colab.research.google.com/github/masondelan/ads504-bitcoin-project/blob/main/ADS_504_Group_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup

In [3]:
import sys
assert sys.version_info >= (3, 7)

from packaging import version
import sklearn
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

# --- Installs (xgboost/lightgbm/pandas-ta aren't preinstalled on Colab) ---
!pip install xgboost lightgbm pandas-ta -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.


# Libraries

In [5]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import requests
import zipfile
import io
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler, Normalizer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.metrics import (
    confusion_matrix, accuracy_score, classification_report,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import sklearn
import xgboost as xgb
import lightgbm as lgb

SEED = 42
np.random.seed(SEED)

plt.rcParams['figure.figsize'] = (12, 5)

In [9]:
url = "https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1h/BTCUSDT-1h-2021-01.zip"

resp = requests.get(url)
with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
    zf.extractall("data/raw")

cols = ["open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "num_trades",
        "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"]

df = pd.read_csv("data/raw/BTCUSDT-1h-2021-01.csv", header=None, names=cols)
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,1609459200000,28923.63,29031.34,28690.17,28995.13,2311.811445,1609462799999,6.676883e+07,58389,1215.359238,3.510354e+07,0
1,1609462800000,28995.13,29470.00,28960.35,29409.99,5403.068471,1609466399999,1.583578e+08,103896,3160.041701,9.261399e+07,0
2,1609466400000,29410.00,29465.26,29120.03,29194.65,2384.231560,1609469999999,6.984265e+07,57646,1203.433506,3.525275e+07,0
3,1609470000000,29195.25,29367.00,29150.02,29278.40,1461.345077,1609473599999,4.276078e+07,42510,775.915666,2.270555e+07,0
4,1609473600000,29278.41,29395.00,29029.40,29220.31,2038.046803,1609477199999,5.961464e+07,55414,1003.342834,2.934638e+07,0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 744 entries, 0 to 743
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   open_time               744 non-null    int64  
 1   open                    744 non-null    float64
 2   high                    744 non-null    float64
 3   low                     744 non-null    float64
 4   close                   744 non-null    float64
 5   volume                  744 non-null    float64
 6   close_time              744 non-null    int64  
 7   quote_asset_volume      744 non-null    float64
 8   num_trades              744 non-null    int64  
 9   taker_buy_base_volume   744 non-null    float64
 10  taker_buy_quote_volume  744 non-null    float64
 11  ignore                  744 non-null    int64  
dtypes: float64(8), int64(4)
memory usage: 69.9 KB
